# 5. Knowledge Distillation
Compares GPT-2 and DistilGPT2 on size, latency and perplexity, then runs a small KL-divergence + cross-entropy distillation demo. The previous BLEU section was removed because its references were invented rather than ground truth.

In [ ]:
# Knowledge distillation: GPT-2 teacher -> DistilGPT2 student
%pip install -q -U transformers datasets accelerate
import time, math, torch
import torch.nn.functional as F
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_name="gpt2"; student_name="distilgpt2"
teacher_tok=AutoTokenizer.from_pretrained(teacher_name); student_tok=AutoTokenizer.from_pretrained(student_name)
teacher_tok.pad_token=teacher_tok.eos_token; student_tok.pad_token=student_tok.eos_token
teacher=AutoModelForCausalLM.from_pretrained(teacher_name).to(device).eval()
student=AutoModelForCausalLM.from_pretrained(student_name).to(device).eval()

def param_mb(m): return sum(p.numel()*p.element_size() for p in m.parameters())/2**20
print("Teacher MB:",param_mb(teacher),"Student MB:",param_mb(student))

# Same held-out text and same generation settings.
prompt="Artificial intelligence is transforming the world of"
def latency(model,tok,n=20,runs=3):
    x=tok(prompt,return_tensors="pt").to(device); vals=[]
    for _ in range(runs):
        if device.type=="cuda": torch.cuda.synchronize()
        t=time.perf_counter(); model.generate(**x,max_new_tokens=n,do_sample=False,pad_token_id=tok.eos_token_id)
        if device.type=="cuda": torch.cuda.synchronize()
        vals.append(time.perf_counter()-t)
    return float(sum(vals)/len(vals))
print("Teacher latency:",latency(teacher,teacher_tok)); print("Student latency:",latency(student,student_tok))

# Proper causal-LM perplexity: no padding, sliding windows, only new target tokens counted.
def perplexity(model,tok,text,max_length=512,stride=256):
    ids=tok(text,return_tensors="pt",add_special_tokens=False).input_ids[0]; ctx=min(max_length,getattr(model.config,"n_positions",1024)); total_nll=0.; total=0
    for end in range(stride,len(ids)+stride,stride):
        end=min(end,len(ids)); begin=max(0,end-ctx); x=ids[begin:end].unsqueeze(0).to(device); trg_len=end-max(stride,begin) if end<len(ids) else end-max(stride,begin)
        if begin==0: trg_len=end
        labels=x.clone(); labels[:,:-trg_len]=-100
        with torch.no_grad(): loss=model(x,labels=labels).loss
        total_nll+=float(loss)*trg_len; total+=trg_len
        if end==len(ids): break
    return math.exp(total_nll/total)
text="Artificial intelligence and machine learning are fields of computer science. "*150
print("Teacher PPL:",perplexity(teacher,teacher_tok,text)); print("Student PPL:",perplexity(student,student_tok,text))

# Mini distillation step. Student and teacher share GPT-2 BPE vocabulary, so logits can be compared directly.
train=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",split="train[:1%]").filter(lambda x:x["text"].strip()!="")
opt=AdamW(student.parameters(),lr=5e-5); T=2.0
student.train(); teacher.eval()
for step,row in enumerate(train.select(range(min(10,len(train))))):
    enc=student_tok(row["text"],return_tensors="pt",truncation=True,max_length=128).to(device)
    with torch.no_grad(): tlog=teacher(**enc).logits
    slog=student(**enc).logits
    # Align next-token distributions.
    kl=F.kl_div(F.log_softmax(slog[:,:-1]/T,dim=-1),F.softmax(tlog[:,1:]/T,dim=-1),reduction="batchmean")*(T*T)
    ce=F.cross_entropy(slog[:,:-1].reshape(-1,slog.size(-1)),enc.input_ids[:,1:].reshape(-1))
    loss=0.5*kl+0.5*ce; loss.backward(); opt.step(); opt.zero_grad(); print(step+1,float(loss))
